In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import sqlite3

# 1. Dynamic Path Resolution
BASE_DIR = Path.cwd().parent 
RAW_DIR = BASE_DIR / "data" / "raw"
PROCESSED_DIR = BASE_DIR / "data" / "processed"
DB_DIR = BASE_DIR / "data" / "db"

# --- 1. NAV DATA CLEANING ---
nav_df = pd.read_csv(RAW_DIR / "02_nav_history.csv")
nav_df['date'] = pd.to_datetime(nav_df['date'])
nav_df['amfi_code'] = pd.to_numeric(nav_df['amfi_code'], errors='coerce').astype('Int64').astype(str)

# Sorting and deduplication
nav_df = nav_df.sort_values(by=['amfi_code', 'date']).drop_duplicates(subset=['amfi_code', 'date'])
nav_df = nav_df[nav_df['nav'] > 0]

print("NAV Rows before business-day fill:", len(nav_df))

# Time-Series Sampling (Handling weekends/holidays securely)
nav_df = nav_df.set_index('date')
nav_df = nav_df.groupby('amfi_code')['nav'].resample('B').ffill().reset_index()
print("NAV Rows after business-day fill:", len(nav_df))

nav_df.to_csv(PROCESSED_DIR / "clean_nav.csv", index=False)

NAV Rows before business-day fill: 46000
NAV Rows after business-day fill: 46000


In [2]:
# --- 2. TRANSACTIONS DATA CLEANING ---
tx_df = pd.read_csv(RAW_DIR / "08_investor_transactions.csv")
tx_df['transaction_date'] = pd.to_datetime(tx_df['transaction_date'])
tx_df['transaction_type'] = tx_df['transaction_type'].str.strip().str.title().replace({'Sip': 'SIP'})
tx_df = tx_df[tx_df['amount_inr'] > 0]

valid_kyc = ['Verified', 'Pending']
invalid_kyc_count = len(tx_df[~tx_df['kyc_status'].isin(valid_kyc)])
print(f"Found {invalid_kyc_count} rows with invalid KYC status. Filtering them out.")
tx_df = tx_df[tx_df['kyc_status'].isin(valid_kyc)]

tx_df.to_csv(PROCESSED_DIR / "clean_transactions.csv", index=False)

Found 0 rows with invalid KYC status. Filtering them out.


In [3]:
# --- 3. FUND MASTER DATA CLEANING ---
fund_df = pd.read_csv(RAW_DIR / "01_fund_master.csv")
initial_len = len(fund_df)
fund_df = fund_df[(fund_df['expense_ratio_pct'] >= 0.1) & (fund_df['expense_ratio_pct'] <= 2.5)]
print(f"Dropped {initial_len - len(fund_df)} funds due to invalid expense ratios.")

fund_df.to_csv(PROCESSED_DIR / "clean_fund_master.csv", index=False)

Dropped 0 funds due to invalid expense ratios.


In [4]:
# --- 4. PERFORMANCE DATA CLEANING ---
perf_df = pd.read_csv(RAW_DIR / "07_scheme_performance.csv")
return_cols = ['return_1yr_pct', 'return_3yr_pct', 'return_5yr_pct']
for col in return_cols:
    perf_df[col] = pd.to_numeric(perf_df[col], errors='coerce')

perf_df['negative_sharpe_flag'] = perf_df['sharpe_ratio'] < 0
negative_sharpe_count = perf_df['negative_sharpe_flag'].sum()
print(f"Flagged {negative_sharpe_count} funds with a negative Sharpe Ratio.")

perf_df.to_csv(PROCESSED_DIR / "clean_performance.csv", index=False)

Flagged 0 funds with a negative Sharpe Ratio.


In [5]:
# --- 5. AUM DATA CLEANING ---
print("Cleaning AUM Data...")
aum_df = pd.read_csv(RAW_DIR / "03_aum_by_fund_house.csv")
aum_df['date'] = pd.to_datetime(aum_df['date'])
aum_df['aum_crore'] = pd.to_numeric(aum_df['aum_crore'], errors='coerce')
aum_df = aum_df[aum_df['aum_crore'] > 0] 
aum_df.to_csv(PROCESSED_DIR / "clean_aum.csv", index=False)

Cleaning AUM Data...


In [6]:
from sqlalchemy import create_engine

BASE_DIR = Path.cwd().parent
PROCESSED_DIR = BASE_DIR / "data" / "processed"
DB_DIR = BASE_DIR / "data" / "db"
db_path = DB_DIR / "bluestock_mf.db"

# 1. Create the SQLAlchemy Engine
engine = create_engine(f'sqlite:///{db_path}')

# 2. Load the processed CSVs
fund_df = pd.read_csv(PROCESSED_DIR / "clean_fund_master.csv")
nav_df = pd.read_csv(PROCESSED_DIR / "clean_nav.csv")
tx_df = pd.read_csv(PROCESSED_DIR / "clean_transactions.csv")
perf_df = pd.read_csv(PROCESSED_DIR / "clean_performance.csv")

# Renaming 'date' to 'nav_date' in nav_df to match schema
if 'date' in nav_df.columns:
    nav_df = nav_df.rename(columns={'date': 'nav_date'})

# 3. Generate dim_date dynamically
print("Generating dim_date...")
start_date = '2022-01-01'
end_date = '2026-05-31'
date_range = pd.date_range(start=start_date, end=end_date, freq='D')

dim_date = pd.DataFrame({'date': date_range})
dim_date['date_id'] = dim_date['date'].dt.strftime('%Y%m%d').astype(int)
dim_date['year'] = dim_date['date'].dt.year
dim_date['month'] = dim_date['date'].dt.month
dim_date['quarter'] = dim_date['date'].dt.quarter
dim_date['is_weekday'] = dim_date['date'].dt.dayofweek < 5 
dim_date = dim_date[['date_id', 'date', 'year', 'month', 'quarter', 'is_weekday']]
dim_date['date'] = dim_date['date'].dt.strftime('%Y-%m-%d')

# 4. Load datasets into SQLite
print("Loading all tables into SQLite...")
fund_df.to_sql('dim_fund', engine, if_exists='replace', index=False)
dim_date.to_sql('dim_date', engine, if_exists='replace', index=False)
nav_df.to_sql('fact_nav', engine, if_exists='replace', index=False)
tx_df.to_sql('fact_transactions', engine, if_exists='replace', index=False)
perf_df.to_sql('fact_performance', engine, if_exists='replace', index=False)
aum_df.to_sql('fact_aum', engine, if_exists='replace', index=False)

print("Database successfully loaded! You are ready for Day 3.")

Generating dim_date...
Loading all tables into SQLite...
Database successfully loaded! You are ready for Day 3.
